# Results analysis (local)

Analyze checkpoints copied from Kaggle into **`results/`** (no training here).

**Expected layout:**
```
results/
  stage_1_vae/
    model.pth
    samples_epoch_10.png
    ...
  stage_1_dcgan/
    ...
  reports/          (optional — FID CSVs from Kaggle)
    fid_stage1.csv
    fid_stage2_hyperparams.csv
```

New figures and CSVs from this notebook go to **`outputs/`**.

In [ ]:
import os
import torch

# --- paths (edit if needed) ---
RESULTS_DIR = "./results/"
DATA_DIR = "./data/cats"  # only needed if you recompute FID locally

import utils.config as cfg
cfg.DATA_DIR = DATA_DIR
cfg.NUM_WORKERS = 0

from utils.config import (
    EXPERIMENTS,
    FID_NUM_SAMPLES,
    OUTPUT_DIR,
    STAGE_1_SCENARIOS,
    STAGE_2_SCENARIOS,
    STAGE_4_SCENARIOS,
    seed_everything,
)
from utils.analysis_utils import (
    best_scenario_by_fid,
    list_result_scenarios,
    list_sample_checkpoints,
    load_fid_csv,
    print_fid_rows,
    recommend_for_stage3,
    scenario_dir,
    show_checkpoint_progression,
    show_saved_image,
)
from utils.evaluation_utils import (
    compare_cats_vs_cats_dogs,
    compare_fid_scenarios,
    compare_mode_collapse_scenarios,
    generate_candidate_grid,
    interpolate_between_latents,
    load_trained_model,
    output_path,
    print_fid_table,
    save_interpolation_grid,
    save_latent_codes,
    save_sample_grid,
    show_image_grid,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything()
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "reports"), exist_ok=True)

print("Device:", device)
print("Scenarios with model.pth:", list_result_scenarios(RESULTS_DIR))

## Stage 1 — Baseline (VAE vs DCGAN)

1. Visual: training grids every 10 epochs  
2. FID table (from saved CSV **or** recompute if you have `data/cats` locally)

In [ ]:
for name in STAGE_1_SCENARIOS:
    if name in list_result_scenarios(RESULTS_DIR):
        show_checkpoint_progression(name, RESULTS_DIR)

In [ ]:
# Option A: load FID table from Kaggle (fast, no GPU work)
fid_stage1 = load_fid_csv(os.path.join(RESULTS_DIR, "reports", "fid_stage1.csv"))
print_fid_rows(fid_stage1)

# Option B: recompute FID (needs DATA_DIR + GPU time) — uncomment:
# fid_stage1 = compare_fid_scenarios(
#     STAGE_1_SCENARIOS,
#     device,
#     max_samples=FID_NUM_SAMPLES,
#     models_dir=RESULTS_DIR,
#     save_csv=os.path.join(OUTPUT_DIR, "reports", "fid_stage1.csv"),
# )
# print_fid_table(fid_stage1)

In [ ]:
# Final samples (epoch 50) — side by side for the report
for name in STAGE_1_SCENARIOS:
    paths = list_sample_checkpoints(name, RESULTS_DIR)
    if paths:
        show_saved_image(paths[-1], title=f"Final samples — {name}")

## Stage 2 — Hyperparameters

Compare FID and sample grids, then decide what to use before stage 3.

In [ ]:
for name in STAGE_2_SCENARIOS:
    if name in list_result_scenarios(RESULTS_DIR):
        show_checkpoint_progression(name, RESULTS_DIR)

In [ ]:
fid_stage2 = load_fid_csv(os.path.join(RESULTS_DIR, "reports", "fid_stage2_hyperparams.csv"))
print_fid_rows(fid_stage2)

# Uncomment to recompute locally:
# fid_stage2 = compare_fid_scenarios(
#     STAGE_2_SCENARIOS, device, models_dir=RESULTS_DIR,
#     save_csv=os.path.join(OUTPUT_DIR, "reports", "fid_stage2_hyperparams.csv"),
# )

In [ ]:
# --- Decision for stage 3 ---
rec = recommend_for_stage3(fid_stage1, fid_stage2)
print("Recommended for interpolation:")
print("  VAE:  ", rec["vae_for_interpolation"])
print("  DCGAN:", rec["dcgan_for_interpolation"])
if "best_vae_stage2_by_fid" in rec:
    print("  Best VAE stage-2 (subset data):", rec["best_vae_stage2_by_fid"])
    print("  Best DCGAN stage-2 (subset data):", rec["best_dcgan_stage2_by_fid"])
for note in rec["notes"]:
    print(" -", note)

## Stage 3 — Latent interpolation

Uses checkpoints from **`results/`**. Pick two indices from the candidate grid.

In [ ]:
SCENARIO = "stage_1_vae"  # change from recommendation above

model, config, _ = load_trained_model(SCENARIO, device, models_dir=RESULTS_DIR)
display_imgs, latents, _ = generate_candidate_grid(model, config, device, num_candidates=64, seed=123)
show_image_grid(display_imgs, title=f"Pick IDX_A and IDX_B — {SCENARIO}", nrow=8)
save_sample_grid(display_imgs, output_path(SCENARIO, "qualitative_grid.png"), nrow=8)

In [ ]:
IDX_A = 3
IDX_B = 47

z_a = latents[IDX_A].clone()
z_b = latents[IDX_B].clone()
save_latent_codes(z_a, z_b, output_path(SCENARIO, "interpolation_latents.pth"))

interp_display, interp_raw, is_dcgan = interpolate_between_latents(
    model, config, z_a, z_b, device, num_steps=8
)
save_interpolation_grid(interp_raw, output_path(SCENARIO, "interpolation_10_images.png"), is_dcgan=is_dcgan)
show_image_grid(interp_display, title=f"Interpolation {IDX_A} → {IDX_B} (10 images)", nrow=10)

## Stage 4 — Mode collapse (DCGAN)

Run after you have `stage_1_dcgan` and optional `stage_4_*` folders in `results/`.

In [ ]:
stage4_available = [s for s in STAGE_4_SCENARIOS if s in list_result_scenarios(RESULTS_DIR)]
print("Available for mode-collapse check:", stage4_available)

if stage4_available:
    collapse_results = compare_mode_collapse_scenarios(
        stage4_available, device, models_dir=RESULTS_DIR
    )

In [ ]:
# Compare sample grids visually
for name in stage4_available:
    paths = list_sample_checkpoints(name, RESULTS_DIR)
    if paths:
        show_saved_image(paths[-1], title=name)

## Stage 5 — Cats + dogs vs cats only

Needs `stage_5_*` and matching `stage_1_*` in `results/`, plus local data if recomputing FID.

In [ ]:
SCENARIO_CATS_ONLY = "stage_1_dcgan"
SCENARIO_MIXED = "stage_5_dcgan_cats_dogs"

if SCENARIO_MIXED in list_result_scenarios(RESULTS_DIR):
    summary, imgs_cat, imgs_mix = compare_cats_vs_cats_dogs(
        SCENARIO_CATS_ONLY,
        SCENARIO_MIXED,
        device,
        models_dir=RESULTS_DIR,
        save_dir=os.path.join(OUTPUT_DIR, "stage_5_comparison"),
    )
    show_image_grid(imgs_cat, title="Cats only", nrow=8)
    show_image_grid(imgs_mix, title="Cats + dogs", nrow=8)
else:
    print(f"Put trained {SCENARIO_MIXED} under {RESULTS_DIR} first.")

## Report checklist

- Stage 1: FID table, VAE vs DCGAN final grids, sharpness vs blur comment  
- Stage 2: which hyperparameters helped (FID + visuals); note 25% subset  
- Stage 3: 10-image interpolation + latent codes saved; smoothness discussion  
- Stage 4: mode collapse yes/no + mitigation if trained  
- Stage 5: distinct cats/dogs vs blended samples (exploratory)  

Files for the report: `results/` (training artifacts) + `outputs/` (this notebook).